In [ ]:
%pip install -U mlflow pandas

dbutils.library.restartPython()


In [0]:
import json
import time
import mlflow
import pandas as pd

CONFIG_PATH = "dbfs:/Volumes/main/default/dino_pdfs/project/config/project_config.json"
config = json.loads(dbutils.fs.head(CONFIG_PATH, 100000))

MLFLOW_EXPERIMENT = config["mlflow_experiment"]
TOP_K = int(config.get("top_k", 4))

mlflow.set_experiment(MLFLOW_EXPERIMENT)

print("Loaded config successfully.")
print(json.dumps(config, indent=2))


Loaded config successfully.
{
  "project_name": "DinoRAG",
  "domain_name": "Dinosaurs",
  "catalog": "main",
  "schema": "default",
  "volume": "dino_pdfs",
  "volume_uri": "/Volumes/main/default/dino_pdfs",
  "project_uri": "/Volumes/main/default/dino_pdfs/project",
  "artifacts_uri": "/Volumes/main/default/dino_pdfs/project/artifacts",
  "index_uri": "/Volumes/main/default/dino_pdfs/project/faiss_index",
  "config_uri": "/Volumes/main/default/dino_pdfs/project/config",
  "mlflow_experiment": "/Shared/Dinosaur_RAG_Experiments",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "top_k": 4,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "chat_endpoint": "databricks-meta-llama-3-1-8b-instruct",
  "delta_table": "main.default.dinosaur_chunks"
}


In [0]:
evaluation_questions = [
    "What did Triceratops eat?",
    "Which period did Stegosaurus live in?",
    "Explain one dinosaur extinction theory.",
    "Compare T-Rex and Velociraptor.",
    "Which dinosaur lived on Mars?"
]

evaluation_questions


['What did Triceratops eat?',
 'Which period did Stegosaurus live in?',
 'Explain one dinosaur extinction theory.',
 'Compare T-Rex and Velociraptor.',
 'Which dinosaur lived on Mars?']

In [0]:
%run /Dinosaur_RAG_Project/04_rag_pipeline_with_memory_tools

  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached mlflow-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached langgraph-1.1.8-py3-none-any.whl.metadata (8.0 kB)
  Using cached langchain_classic-1.0.4-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looking at multiple versions of langchain-databricks to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_databricks-0.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_databricks-0.1.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_databricks-0.1.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
INFO: pip is still looking at multiple versions of langchain-databricks to determine which version is compatible with oth

Loaded config:
{
  "project_name": "DinoRAG",
  "domain_name": "Dinosaurs",
  "catalog": "main",
  "schema": "default",
  "volume": "dino_pdfs",
  "volume_uri": "/Volumes/main/default/dino_pdfs",
  "project_uri": "/Volumes/main/default/dino_pdfs/project",
  "artifacts_uri": "/Volumes/main/default/dino_pdfs/project/artifacts",
  "index_uri": "/Volumes/main/default/dino_pdfs/project/faiss_index",
  "config_uri": "/Volumes/main/default/dino_pdfs/project/config",
  "mlflow_experiment": "/Shared/Dinosaur_RAG_Experiments",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "top_k": 4,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "chat_endpoint": "databricks-meta-llama-3-1-8b-instruct",
  "delta_table": "main.default.dinosaur_chunks"
}


MLflow LangChain autolog enabled.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index loaded successfully.
Top-k: 4


Chat endpoint configured: databricks-meta-llama-3-1-8b-instruct


In [0]:
import time

results = []

top_k_values = [3, 4, 5]

for k in top_k_values:
    with mlflow.start_run(run_name=f"04_eval_top_k_{k}"):
        mlflow.log_param("top_k", k)

        per_k_rows = []

        for q in evaluation_questions:
            start_time = time.time()
            result = ask_dino_bot(q, k=k, save_memory=False)
            latency = time.time() - start_time

            row = {
                "question": q,
                "top_k": k,
                "latency_sec": round(latency, 4),
                "answer_length": len(result["answer"]),
                "mode": result["mode"],
                "label": result["label"],
                "source_count": len(result["sources"]),
                "avg_retrieval_score": round(
                    sum(result.get("retrieval_scores", [])) / len(result.get("retrieval_scores", [])),
                    4
                ) if result.get("retrieval_scores") else None,
                "answer_preview": result["answer"][:300]
            }

            results.append(row)
            per_k_rows.append(row)

        avg_latency = sum(r["latency_sec"] for r in per_k_rows) / len(per_k_rows)
        avg_answer_length = sum(r["answer_length"] for r in per_k_rows) / len(per_k_rows)
        pdf_mode_count = sum(1 for r in per_k_rows if r["mode"] == "pdf")
        external_mode_count = sum(1 for r in per_k_rows if r["mode"] == "external")

        mlflow.log_metric("avg_latency_sec", avg_latency)
        mlflow.log_metric("avg_answer_length", avg_answer_length)
        mlflow.log_metric("pdf_mode_count", pdf_mode_count)
        mlflow.log_metric("external_mode_count", external_mode_count)

print("Evaluation runs completed.")

Evaluation runs completed.


QUESTION: Tell me about Triceratops.
Answer source: PDF Knowledge Base
Triceratops is mentioned in the context of the definition of Dinosauria. According to the definition, the most recent common ancestor (MRCA) of Triceratops and modern birds (Neornithes) is used to define the group Dinosauria, and all its descendants.
Sources: [{'source_file': 'Document6_Dinosaur_Reproduction_Parental_Care_and_Migration Behavior.pdf', 'page': 1, 'chunk_id': 126}, {'source_file': 'Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'page': 1, 'chunk_id': 5}, {'source_file': 'Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'page': 9, 'chunk_id': 37}]
Scores: [1.172004222869873, 1.2316474914550781, 1.232895851135254, 1.2472107410430908]


QUESTION: What did Stegosaurus eat?
Answer source: PDF Knowledge Base
I could not find enough information in the PDF knowledge base about what Stegosaurus ate.
Sources: [{'source_file': 'Document3_Dinosaur_Classification_Size_and_Behavior.pdf', 'page': 4, 'chunk_id': 75}, {'source_file': 'Document8_Common_Misconceptions_About_Dinosaurs.pdf', 'page': 3, 'chunk_id': 165}, {'source_file': 'Document3_Dinosaur_Classification_Size_and_Behavior.pdf', 'page': 1, 'chunk_id': 61}]
Scores: [0.7887638807296753, 0.8011966347694397, 0.9182360172271729, 0.923947811126709]


QUESTION: Which dinosaur lived on Mars?
Answer source: PDF Knowledge Base
I could not find enough information in the PDF knowledge base.
Sources: [{'source_file': 'Document8_Common_Misconceptions_About_Dinosaurs.pdf', 'page': 2, 'chunk_id': 162}, {'source_file': 'Document3_Dinosaur_Classification_Size_and_Behavior.pdf', 'page': 1, 'chunk_id': 64}, {'source_file': 'Document8_Common_Misconceptions_About_Dinosaurs.pdf', 'page': 3, 'chunk_id': 166}]
Scores: [0.9336845874786377, 1.0095248222351074, 1.0096521377563477, 1.0150883197784424]


Notebook 4 completed successfully.


In [0]:
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)


question,top_k,latency_sec,answer_length,mode,label,source_count,avg_retrieval_score,answer_preview
What did Triceratops eat?,3,1.1546,62,pdf,Answer source: PDF Knowledge Base,3,1.1388,I could not find enough information in the PDF knowledge base.
Which period did Stegosaurus live in?,3,1.0242,54,pdf,Answer source: PDF Knowledge Base,3,0.7986,"The late Jurassic period, about 150 million years ago."
Explain one dinosaur extinction theory.,3,1.273,203,pdf,Answer source: PDF Knowledge Base,3,0.7089,"One dinosaur extinction theory is the extraterrestrial impact event in the Yucatán Peninsula, which is thought to have played a major role in the mass extinction event that occurred 65 million years ago."
Compare T-Rex and Velociraptor.,3,1.535,327,pdf,Answer source: PDF Knowledge Base,3,0.951,"Tyrannosaurus (T-Rex) and Velociraptor lived in different time periods. Tyrannosaurus evolved around 67 million years ago, during the Cretaceous period, while Velociraptor is not mentioned in the provided context. However, it is mentioned that a Velociraptor fossil was discovered in 1971, showing it"
Which dinosaur lived on Mars?,3,1.0086,62,pdf,Answer source: PDF Knowledge Base,3,0.9843,I could not find enough information in the PDF knowledge base.
What did Triceratops eat?,4,1.1018,92,pdf,Answer source: PDF Knowledge Base,4,1.1453,I could not find enough information in the PDF knowledge base about the diet of Triceratops.
Which period did Stegosaurus live in?,4,1.0744,54,pdf,Answer source: PDF Knowledge Base,4,0.8194,"The late Jurassic period, about 150 million years ago."
Explain one dinosaur extinction theory.,4,1.2499,203,pdf,Answer source: PDF Knowledge Base,4,0.7305,"One dinosaur extinction theory is the extraterrestrial impact event in the Yucatán Peninsula, which is thought to have played a major role in the mass extinction event that occurred 65 million years ago."
Compare T-Rex and Velociraptor.,4,1.1494,96,pdf,Answer source: PDF Knowledge Base,4,0.961,I could not find enough information in the PDF knowledge base to compare T-Rex and Velociraptor.
Which dinosaur lived on Mars?,4,1.0406,62,pdf,Answer source: PDF Knowledge Base,4,0.992,I could not find enough information in the PDF knowledge base.


In [0]:
ARTIFACTS_URI = config["artifacts_uri"]
output_path = "/Volumes/main/default/dino_pdfs/project/artifacts/eval_results.csv"

results_df.to_csv(output_path, index=False)
print("Saved evaluation results to:", output_path)

Saved evaluation results to: /Volumes/main/default/dino_pdfs/project/artifacts/eval_results.csv


In [0]:
final_demo_questions = [
    "Tell me about Triceratops.",
    "Was it a herbivore?",
    "Compare Triceratops and Tyrannosaurus rex.",
    "Explain one dinosaur extinction theory.",
    "Which dinosaur lived on Mars?"
]

demo_rows = []

for q in final_demo_questions:
    result = ask_dino_bot(q, k=TOP_K, save_memory=False)
    demo_rows.append({
        "question": q,
        "mode": result["mode"],
        "label": result["label"],
        "source_count": len(result["sources"]),
        "answer_preview": result["answer"][:250]
    })

demo_df = pd.DataFrame(demo_rows)
display(demo_df)


question,mode,label,source_count,answer_preview
Tell me about Triceratops.,pdf,Answer source: PDF Knowledge Base,4,"Triceratops is mentioned in the context of the definition of Dinosauria. According to the text, Triceratops is one of the species used to define the most recent common ancestor (MRCA) of dinosaurs, along with modern birds (Neornithes)."
Was it a herbivore?,pdf,Answer source: PDF Knowledge Base,4,I could not find enough information in the PDF knowledge base to determine if the dinosaur mentioned was a herbivore.
Compare Triceratops and Tyrannosaurus rex.,pdf,Answer source: PDF Knowledge Base,4,I could not find enough information in the PDF knowledge base to compare Triceratops and Tyrannosaurus rex.
Explain one dinosaur extinction theory.,pdf,Answer source: PDF Knowledge Base,4,"One dinosaur extinction theory is the extraterrestrial impact event in the Yucatán Peninsula, which is thought to have played a major role in the mass extinction event that occurred 65 million years ago."
Which dinosaur lived on Mars?,pdf,Answer source: PDF Knowledge Base,4,I could not find enough information in the PDF knowledge base.
